# 05 — XGBoost Model
Task 6: train XGBoost on the Task 5 feature set and compare it honestly against the Task 4 Linear Regression baseline.

## 1. Load the feature-engineered dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/processed/tomato_features.csv"
)

df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)

df.head()

## 2. Check the dataset

In [ ]:
print(df.info())

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.columns.tolist())

## 3. Sort chronologically
Already sorted in Task 5, but re-sort here for safety.

In [ ]:
df = df.sort_values(
    ["date", "market"]
).reset_index(drop=True)

## 4. Define target and features
`market` is categorical text (e.g. Bhubaneswar, Cuttack, Puri) and is intentionally excluded from FEATURES for this first model — proper categorical encoding comes later.

In [ ]:
TARGET = "modal_price"

FEATURES = [
    "min_price",
    "max_price",
    "average_price",
    "price_range",

    "year",
    "month",
    "day",
    "day_of_week",

    "month_sin",
    "month_cos",

    "price_lag_1",
    "price_lag_7",
    "price_lag_14",
    "price_lag_30",

    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",

    "rolling_std_7",

    "price_change_1",
    "price_change_pct"
]

if "arrival_quantity" in df.columns:
    FEATURES += [
        "arrival_quantity",
        "arrival_lag_1",
        "arrival_rolling_7"
    ]

print("Number of features:", len(FEATURES))

## 5. Remove incomplete rows

In [ ]:
model_df = df[
    FEATURES + [TARGET, "date", "market"]
].copy()

model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

model_df = model_df.dropna(
    subset=FEATURES + [TARGET]
)

print("Final modeling data:", model_df.shape)

## 6. Time-based split
No random split — oldest 80% trains, newest 20% tests.

In [ ]:
model_df = model_df.sort_values(
    "date"
).reset_index(drop=True)

split_index = int(
    len(model_df) * 0.8
)

train = model_df.iloc[
    :split_index
]

test = model_df.iloc[
    split_index:
]

print("Training:", train.shape)
print("Testing:", test.shape)

print(
    "Training period:",
    train["date"].min(),
    "→",
    train["date"].max()
)

print(
    "Testing period:",
    test["date"].min(),
    "→",
    test["date"].max()
)

## 7. Create X and y

In [ ]:
X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## 8. Train XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

## 9. Generate predictions

In [ ]:
xgb_predictions = xgb_model.predict(
    X_test
)

print(xgb_predictions[:10])

## 10. Evaluate XGBoost

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

xgb_mae = mean_absolute_error(
    y_test,
    xgb_predictions
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_predictions
    )
)

xgb_mape = np.mean(
    np.abs(
        (y_test - xgb_predictions)
        / y_test
    )
) * 100

xgb_r2 = r2_score(
    y_test,
    xgb_predictions
)

print("XGBoost Performance")
print("-------------------")
print(f"MAE  : {xgb_mae:.2f}")
print(f"RMSE : {xgb_rmse:.2f}")
print(f"MAPE : {xgb_mape:.2f}%")
print(f"R²   : {xgb_r2:.4f}")

## 11. Compare with Linear Regression

In [ ]:
baseline = pd.read_csv(
    "../data/processed/baseline_predictions.csv"
)

baseline_mae = mean_absolute_error(
    baseline["actual_price"],
    baseline["predicted_price"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline["actual_price"],
        baseline["predicted_price"]
    )
)

baseline_mape = np.mean(
    np.abs(
        (
            baseline["actual_price"]
            - baseline["predicted_price"]
        )
        / baseline["actual_price"]
    )
) * 100

baseline_r2 = r2_score(
    baseline["actual_price"],
    baseline["predicted_price"]
)

comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "XGBoost"
    ],

    "MAE": [
        baseline_mae,
        xgb_mae
    ],

    "RMSE": [
        baseline_rmse,
        xgb_rmse
    ],

    "MAPE": [
        baseline_mape,
        xgb_mape
    ],

    "R2": [
        baseline_r2,
        xgb_r2
    ]
})

comparison

## 12. Interpret the comparison
Lower is better for MAE, RMSE, MAPE. Higher is generally better for R².

Ideally: XGBoost MAE/RMSE/MAPE < Linear Regression's, and XGBoost R² > Linear Regression's.

**Report the numbers honestly.** If Linear Regression wins on this dataset, say so and investigate why (e.g. too little history, noisy synthetic-style data, overfitting) rather than adjusting the comparison to look better.

## 13. Actual vs predicted graph

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    xgb_predictions,
    label="XGBoost Predicted"
)

plt.xlabel("Test Samples")
plt.ylabel("Modal Price")
plt.title(
    "XGBoost: Actual vs Predicted Price"
)

plt.legend()
plt.tight_layout()
plt.show()

## 14. Feature importance 🔥
Useful for the SIH presentation — let the model tell you what matters, don't assume it beforehand.

In [ ]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURES
).sort_values(
    ascending=False
)

print(importance)

In [ ]:
plt.figure(figsize=(10, 6))

importance.head(15).sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(
    "Top Features Used by XGBoost"
)

plt.tight_layout()
plt.show()

## 15. Save the XGBoost model

In [ ]:
import os
import joblib

os.makedirs(
    "../models",
    exist_ok=True
)

joblib.dump(
    xgb_model,
    "../models/xgboost_price_model.pkl"
)

print(
    "XGBoost model saved successfully."
)

## 16. Save predictions

In [ ]:
xgb_results = pd.DataFrame({
    "date": test["date"].values,
    "market": test["market"].values,
    "actual_price": y_test.values,
    "predicted_price": xgb_predictions
})

xgb_results["absolute_error"] = np.abs(
    xgb_results["actual_price"]
    - xgb_results["predicted_price"]
)

xgb_results.to_csv(
    "../data/processed/xgboost_predictions.csv",
    index=False
)

print("Predictions saved.")

## 17. Save model comparison

In [ ]:
comparison.to_csv(
    "../data/processed/model_comparison.csv",
    index=False
)

print("Model comparison saved.")

## Next step
Once you have `model_comparison.csv` and know which model actually wins on your real data, Task 7 wires the winning model into the FastAPI backend (`ml/`  →  `backend/`) so it can serve live predictions, and starts building the Net Realization Engine on top of its output.